# Online Grooming Detection — Full Training Pipeline
**Paper:** Street et al. (2025) — *Enhanced Online Grooming Detection Employing Context Determination and Message-Level Analysis*

**Dataset:** PervertedJustice (Koutli, 2023)

---
### Setup Instructions
1. Runtime → Change runtime type → **T4 GPU**
2. Run **Cell 1** (installs packages)
3. Run **Cell 2** (clones dataset from GitHub)
4. Run all remaining cells in order

> ⚠️ **Important:** This notebook classifies *speaker roles* (Adult vs. Child language). It does NOT classify individual messages as grooming content.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers==4.44.2 scikit-learn pandas scipy matplotlib seaborn tqdm

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Clone dataset from GitHub ─────────────────────────────────────
import os

DATASET_PATH = '/content/PervertedJusticeDataset'
RESULTS_PATH = '/content/results'
os.makedirs(RESULTS_PATH, exist_ok=True)

if not os.path.exists(DATASET_PATH):
    !git clone https://github.com/helenkoutli/PervertedJusticeDataset.git {DATASET_PATH}
else:
    print('Dataset already cloned.')

pred_files = os.listdir(f'{DATASET_PATH}/predators')
off_files  = os.listdir(f'{DATASET_PATH}/officers')
print(f'Predator files: {len(pred_files)}')
print(f'Officer files:  {len(off_files)}')

In [ ]:
# ── Cell 3: Data loader ────────────────────────────────────────────────────
from pathlib import Path
import pandas as pd
import numpy as np

def load_dataset(dataset_root):
    """
    Load all predator/officer utterances.
    speaker_role: 1 = predator/Adult, 0 = officer/Child
    Labels come from FILE STRUCTURE only, not message content.
    """
    pred_dir = Path(dataset_root) / 'predators'
    off_dir  = Path(dataset_root) / 'officers'
    records  = []

    for pf in sorted(pred_dir.glob('*.txt')):
        conv_id = pf.stem.replace('predator_', '')
        try:
            lines = [l.strip() for l in pf.read_text(encoding='utf-8', errors='replace').splitlines() if l.strip()]
        except:
            lines = []
        for line in lines:
            records.append({'utterance': line, 'speaker_role': 1, 'conversation_id': conv_id, 'file_name': pf.name})

    for of in sorted(off_dir.glob('*.txt')):
        conv_id = of.stem.replace('officer_', '')
        try:
            lines = [l.strip() for l in of.read_text(encoding='utf-8', errors='replace').splitlines() if l.strip()]
        except:
            lines = []
        for line in lines:
            records.append({'utterance': line, 'speaker_role': 0, 'conversation_id': conv_id, 'file_name': of.name})

    return pd.DataFrame(records)

df = load_dataset(DATASET_PATH)
print(f'Total utterances    : {len(df):,}')
print(f'Predator (Adult=1)  : {(df.speaker_role==1).sum():,}')
print(f'Officer  (Child=0)  : {(df.speaker_role==0).sum():,}')
print(f'Conversations       : {df.conversation_id.nunique()}')
df.head(3)

In [ ]:
# ── Cell 4: Group-based 80/20 split ───────────────────────────────────────
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['conversation_id']))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

# Verify zero leakage
assert len(set(train_df.conversation_id) & set(test_df.conversation_id)) == 0, 'LEAKAGE!'

X_train = train_df['utterance'].tolist()
y_train = train_df['speaker_role'].values
X_test  = test_df['utterance'].tolist()
y_test  = test_df['speaker_role'].values

print(f'Train: {len(train_df):,} utterances | {train_df.conversation_id.nunique()} conversations')
print(f'Test:  {len(test_df):,} utterances  | {test_df.conversation_id.nunique()} conversations')
print(f'Train Adult/Child: {y_train.sum():,} / {(y_train==0).sum():,}')
print(f'Test  Adult/Child: {y_test.sum():,} / {(y_test==0).sum():,}')
print('No leakage confirmed ✓')

In [ ]:
# ── Cell 5: MLA metrics (paper Eq. 1 + TP/IA/IC/O) ────────────────────────
def apply_message_threshold(probabilities, t):
    """Equation (1) from paper: omit if (0.5+t) > n > (0.5-t)"""
    lower = 0.5 - t
    upper = 0.5 + t
    omit_mask = (probabilities > lower) & (probabilities < upper)
    return ~omit_mask, omit_mask

def compute_mla_metrics(y_true, y_pred, n_total=None):
    n_kept   = len(y_true)
    n_total  = n_total or n_kept
    correct  = (y_pred == y_true).sum()
    ia       = ((y_pred == 1) & (y_true == 0)).sum()  # predicted Adult, was Child
    ic       = ((y_pred == 0) & (y_true == 1)).sum()  # predicted Child, was Adult
    return {
        'TP': round(100.0 * correct / n_kept, 1) if n_kept else 0,
        'IA': round(100.0 * ia / n_kept, 1)      if n_kept else 0,
        'IC': round(100.0 * ic / n_kept, 1)      if n_kept else 0,
        'O':  round(100.0 * (n_total - n_kept) / n_total, 1) if n_total else 0,
        'n_total': n_total, 'n_kept': n_kept
    }

print('MLA metric functions defined ✓')

In [ ]:
# ── Cell 6: SVM (full dataset) ─────────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import json, time

print('Training SVM on full dataset...')
t0 = time.time()

svm = Pipeline([
    ('vec', CountVectorizer()),
    ('clf', CalibratedClassifierCV(LinearSVC(max_iter=2000))),
])
svm.fit(X_train, y_train)
print(f'  Trained in {time.time()-t0:.0f}s')

y_pred_svm  = svm.predict(X_test)
svm_metrics = compute_mla_metrics(y_test, y_pred_svm)
print(f'\nSVM MLA Results (full dataset):')
print(f'  TP={svm_metrics["TP"]}%  IA={svm_metrics["IA"]}%  IC={svm_metrics["IC"]}%  O={svm_metrics["O"]}%')

with open(f'{RESULTS_PATH}/mla_svm_full.json', 'w') as f:
    json.dump(svm_metrics, f, indent=2)
print('  Saved to results/mla_svm_full.json')

In [ ]:
# ── Cell 7: Naive Bayes (full dataset) ────────────────────────────────────
from sklearn.naive_bayes import MultinomialNB

print('Training Naive Bayes on full dataset...')
t0 = time.time()

nb = Pipeline([
    ('vec', CountVectorizer()),
    ('clf', MultinomialNB()),
])
nb.fit(X_train, y_train)
print(f'  Trained in {time.time()-t0:.0f}s')

y_pred_nb  = nb.predict(X_test)
nb_metrics = compute_mla_metrics(y_test, y_pred_nb)
print(f'\nNB MLA Results (full dataset):')
print(f'  TP={nb_metrics["TP"]}%  IA={nb_metrics["IA"]}%  IC={nb_metrics["IC"]}%  O={nb_metrics["O"]}%')

with open(f'{RESULTS_PATH}/mla_nb_full.json', 'w') as f:
    json.dump(nb_metrics, f, indent=2)
print('  Saved to results/mla_nb_full.json')

In [ ]:
# ── Cell 8: BERT fine-tuning ───────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN    = 128
BATCH_SIZE = 32   # T4 can handle 32 for BERT
EPOCHS     = 3
LR         = 2e-5

print(f'Device: {DEVICE}')

class UtteranceDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(texts, padding=True, truncation=True,
                                   max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}, self.labels[idx]

def train_transformer(model_name, X_tr, y_tr, X_te, y_te,
                       epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR):
    print(f'\nLoading {model_name}...')
    tokenizer = BertTokenizer.from_pretrained(model_name) \
        if 'bert' in model_name and 'roberta' not in model_name \
        else __import__('transformers').RobertaTokenizer.from_pretrained(model_name)

    ModelClass = BertForSequenceClassification \
        if 'bert' in model_name and 'roberta' not in model_name \
        else __import__('transformers').RobertaForSequenceClassification

    model = ModelClass.from_pretrained(model_name, num_labels=2).to(DEVICE)

    # Encode in batches to avoid OOM during tokenisation
    print(f'  Tokenising {len(X_tr):,} training samples...')
    train_dataset = UtteranceDataset(X_tr, y_tr, tokenizer, MAX_LEN)
    test_dataset  = UtteranceDataset(X_te, y_te, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              pin_memory=True, num_workers=2)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size*2, shuffle=False,
                              pin_memory=True, num_workers=2)

    optimizer   = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1*total_steps),
        num_training_steps=total_steps
    )

    # Training
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch, labels in bar:
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            loss   = model(**batch, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
            bar.set_postfix(loss=f'{total_loss/len(bar):.4f}')

    # Inference — collect P(Adult) for all test messages
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch, _ in tqdm(test_loader, desc='Inference'):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits
            probs  = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)

    return model, tokenizer, np.array(all_probs)

print('Trainer defined ✓')

In [ ]:
# ── Cell 9: Train BERT ─────────────────────────────────────────────────────
bert_model, bert_tok, bert_proba = train_transformer(
    'bert-base-uncased', X_train, y_train, X_test, y_test
)

# Evaluate at all t values
bert_results = {}
print('\nBERT MLA Results (full dataset):')
print(f'{"t":>6}  {"TP%":>7}  {"IA%":>7}  {"IC%":>7}  {"O%":>7}')
print('-' * 42)
for t in [0.0, 0.2, 0.3, 0.4, 0.45]:
    if t == 0.0:
        y_pred  = (bert_proba >= 0.5).astype(int)
        metrics = compute_mla_metrics(y_test, y_pred)
    else:
        kept, _ = apply_message_threshold(bert_proba, t)
        metrics = compute_mla_metrics(y_test[kept], (bert_proba[kept] >= 0.5).astype(int), n_total=len(y_test))
    bert_results[f't={t}'] = metrics
    print(f'{t:>6.2f}  {metrics["TP"]:>7}  {metrics["IA"]:>7}  {metrics["IC"]:>7}  {metrics["O"]:>7}')

with open(f'{RESULTS_PATH}/mla_bert_full.json', 'w') as f:
    json.dump(bert_results, f, indent=2)
print('\nSaved to results/mla_bert_full.json')

# Save model
bert_model.save_pretrained(f'{RESULTS_PATH}/bert_model')
bert_tok.save_pretrained(f'{RESULTS_PATH}/bert_model')
print('BERT model saved ✓')

In [ ]:
# ── Cell 10: Train RoBERTa ─────────────────────────────────────────────────
from transformers import RobertaTokenizer, RobertaForSequenceClassification

roberta_model, roberta_tok, roberta_proba = train_transformer(
    'roberta-base', X_train, y_train, X_test, y_test
)

# Evaluate at all t values
roberta_results = {}
print('\nRoBERTa MLA Results (full dataset):')
print(f'{"t":>6}  {"TP%":>7}  {"IA%":>7}  {"IC%":>7}  {"O%":>7}')
print('-' * 42)
for t in [0.0, 0.2, 0.3, 0.4, 0.45]:
    if t == 0.0:
        y_pred  = (roberta_proba >= 0.5).astype(int)
        metrics = compute_mla_metrics(y_test, y_pred)
    else:
        kept, _ = apply_message_threshold(roberta_proba, t)
        metrics = compute_mla_metrics(y_test[kept], (roberta_proba[kept] >= 0.5).astype(int), n_total=len(y_test))
    roberta_results[f't={t}'] = metrics
    print(f'{t:>6.2f}  {metrics["TP"]:>7}  {metrics["IA"]:>7}  {metrics["IC"]:>7}  {metrics["O"]:>7}')

with open(f'{RESULTS_PATH}/mla_roberta_full.json', 'w') as f:
    json.dump(roberta_results, f, indent=2)
print('\nSaved to results/mla_roberta_full.json')

roberta_model.save_pretrained(f'{RESULTS_PATH}/roberta_model')
roberta_tok.save_pretrained(f'{RESULTS_PATH}/roberta_model')
print('RoBERTa model saved ✓')

In [ ]:
# ── Cell 11: Context Determination ────────────────────────────────────────
from scipy.stats import binomtest

def determine_actor(predictions, t=0.0, ast=0.05):
    """Binomial test → Adult(A) / Child(C) / Not Significant(NS)"""
    if t > 0:
        kept = predictions[~((predictions > 0.5-t) & (predictions < 0.5+t))]
    else:
        kept = predictions
    if len(kept) == 0:
        return 'NS'
    adult_votes = (kept >= 0.5).sum()
    p_val = binomtest(int(adult_votes), len(kept), p=0.5, alternative='two-sided').pvalue
    if p_val < ast:
        return 'A' if kept.mean() > 0.5 else 'C'
    return 'NS'

def context_determination(test_df, proba_array, t=0.0, ast=0.05):
    """Run full context determination sweep over all conversations."""
    test_df = test_df.copy()
    test_df['pred_proba'] = proba_array
    results = []
    for conv_id, grp in test_df.groupby('conversation_id'):
        pred_proba = grp[grp.speaker_role==1]['pred_proba'].values
        off_proba  = grp[grp.speaker_role==0]['pred_proba'].values
        if len(pred_proba)==0 or len(off_proba)==0:
            continue
        a = determine_actor(pred_proba, t, ast)
        b = determine_actor(off_proba,  t, ast)
        is_ac = ((a=='A') ^ (b=='A')) and ((a=='C') ^ (b=='C')) and 'NS' not in (a, b)
        results.append({'conversation_id': conv_id, 'pred_det': a, 'off_det': b,
                        'pred_is_ac': is_ac, 'true_ac': True})
    df_r = pd.DataFrame(results)
    tp = (df_r.pred_is_ac & df_r.true_ac).sum()
    fn = (~df_r.pred_is_ac & df_r.true_ac).sum()
    f1 = (2*tp)/(2*tp+fn) if (2*tp+fn)>0 else 0
    return {'TP': int(tp), 'FN': int(fn), 'FP': 0, 'TN': 0, 'F1': round(f1, 3), 'n': len(df_r)}

AST_VALUES = [0.05, 0.01, 0.001]
T_TRAD     = [0.0]
T_TRANS    = [0.2, 0.3, 0.4, 0.45]

all_ctx = {}
print(f'{"Model":<12} {"t":>5} {"AST":>7} {"TP":>5} {"FN":>5} {"F1":>7}')
print('-' * 45)

for model_name, proba_arr, t_vals in [
    ('SVM',     svm.predict_proba(X_test)[:,1], T_TRAD),
    ('NB',      nb.predict_proba(X_test)[:,1],  T_TRAD),
    ('BERT',    bert_proba,                     T_TRANS),
    ('RoBERTa', roberta_proba,                  T_TRANS),
]:
    all_ctx[model_name] = []
    for t in t_vals:
        for ast in AST_VALUES:
            m = context_determination(test_df, proba_arr, t=t, ast=ast)
            m.update({'Model': model_name, 't': t, 'AST': ast})
            all_ctx[model_name].append(m)
            print(f'{model_name:<12} {t:>5.2f} {ast:>7.3f} {m["TP"]:>5} {m["FN"]:>5} {m["F1"]:>7.3f}')

ctx_df = pd.concat([pd.DataFrame(v) for v in all_ctx.values()])
ctx_df.to_csv(f'{RESULTS_PATH}/context_all_models.csv', index=False)
print('\nSaved to results/context_all_models.csv')

In [ ]:
# ── Cell 12: Results summary + paper comparison ────────────────────────────
import matplotlib.pyplot as plt

PAPER = {
    'SVM':            {'TP': 76.0, 'IA': 13.5, 'IC': 10.5, 'O': 0.0},
    'NB':             {'TP': 74.7, 'IA': 12.0, 'IC': 13.3, 'O': 0.0},
    'BERT_t=0.45':    {'TP': 87.3, 'IA':  7.1, 'IC':  5.6, 'O': 30.0},
    'RoBERTa_t=0.45': {'TP': 88.8, 'IA':  5.3, 'IC':  5.9, 'O': 30.4},
}

print('=' * 60)
print('FINAL RESULTS SUMMARY — Full Dataset')
print('(Speaker-role classification: Adult vs. Child language)')
print('NOT grooming detection accuracy')
print('=' * 60)

our = {
    'SVM':            svm_metrics,
    'NB':             nb_metrics,
    'BERT_t=0.45':    bert_results.get('t=0.45', {}),
    'RoBERTa_t=0.45': roberta_results.get('t=0.45', {}),
}
print(f'{"Model":<20} {"TP (Ours)":>12} {"TP (Paper)":>12}')
print('-' * 46)
for k, v in our.items():
    paper_tp = PAPER.get(k, {}).get('TP', '-')
    print(f'{k:<20} {str(v.get("TP","-")):>12} {str(paper_tp):>12}')

# Bar chart
labels  = list(our.keys())
ours_tp = [our[k].get('TP', 0) for k in labels]
paper_tp= [PAPER[k]['TP'] for k in labels]
x = range(len(labels))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i-0.2 for i in x], ours_tp,  0.35, label='Our Results', color='#3498DB')
ax.bar([i+0.2 for i in x], paper_tp, 0.35, label='Paper (ref)', color='#E67E22', alpha=0.7)
ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel('TP%'); ax.set_ylim(0, 100)
ax.set_title('MLA TP% — Our Results vs. Paper (Speaker-Role Classification)')
ax.legend(); plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/tp_comparison.png', dpi=150)
plt.show()
print('Plot saved ✓')

In [ ]:
# ── Cell 13: Download all results as zip ──────────────────────────────────
import shutil
from google.colab import files

zip_path = '/content/grooming_detection_results'
shutil.make_archive(zip_path, 'zip', RESULTS_PATH)
files.download(f'{zip_path}.zip')
print('Download started — check your browser downloads.')
print('\nFiles in results/:')
for f in sorted(os.listdir(RESULTS_PATH)):
    size = os.path.getsize(f'{RESULTS_PATH}/{f}')
    print(f'  {f} ({size/1024:.0f} KB)')